
@author: Akaqox(Salih KIZILIŞIK)

This script is experimental codes of paper "Explainable Multimodal Machine Learning Model
for Predicting Intensive Care Unit Admission " by S.Kizilisik et al. You may use the codes only for research. 
Please cite the paper if you use any part of the codes.

In [ ]:
"""

@author: Akaqox(Salih KIZILIŞIK)

This script is experimental codes of paper "Explainable Multimodal Machine Learning Model
for Predicting Intensive Care Unit Admission " by S.Kizilisik et al. You may use the codes only for research. 
Plese cite the paper if you use any part of the codes.

"""


import tensorflow as tf
from utils.metrics import f1_score

# 1. Load models without compiling (ignores custom metrics like f1_score)
clinic_model = tf.keras.models.load_model(
    'results/2026-03-22/clinic/models/weights_20260322_125026_24ep_8000img.keras', 
    compile=False
)

imaging_model = tf.keras.models.load_model(
    'model/cnn_170824.keras', 
    compile=False
)

fused_model = tf.keras.models.load_model(
    'results/2026-04-02/multi/models/weights_20260402_151622_52ep_8000img.keras',
    custom_objects={'f1_score': f1_score})
fused_model.get_layer('icu').activation = tf.keras.activations.sigmoid
# 2. Verify the architecture
fused_model.summary()



Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 feature_input (InputLayer)     [(None, 576)]        0           []                               
                                                                                                  
 tabular_input (InputLayer)     [(None, 69)]         0           []                               
                                                                                                  
 encode_features (Dense)        (None, 128)          73856       ['feature_input[0][0]']          
                                                                                                  
 encode_tabular (Dense)         (None, 64)           4480        ['tabular_input[0][0]']          
                                                                                              

In [19]:
print(f"ICU Layer Activation: {fused_model.get_layer('icu').activation.__name__}")
print(f"ICU Layer Activation: {fused_model.get_layer('dense1').activation.__name__}")

ICU Layer Activation: sigmoid
ICU Layer Activation: relu


In [20]:
import tensorflow as tf


# 1. Define the input layers (must match the shapes your models expect)
image_input = imaging_model.input  # (None, 224, 224, 3)
# Assuming fused_model input 1 is clinic and input 0 is imaging features (check summary)
clinic_input = tf.keras.Input(shape=(69,), name="clinic_input") 

# 2. Get the feature vector (576 dims) from the imaging backbone
# We use the -2nd layer specifically as you requested
img_features = imaging_model.layers[-3].output

# 3. Pass both into the fused_model
# Ensure the list order [img_features, clinic_input] matches your fused_model's training
final_output = fused_model([img_features, clinic_input])

# 4. Create the final combined model
end_to_end_model = tf.keras.Model(
    inputs=[image_input, clinic_input], 
    outputs=final_output
)

end_to_end_model.summary()

Model: "model_6"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 Conv (Conv2D)                  (None, 112, 112, 16  432         ['input_1[0][0]']                
                                )                                                                 
                                                                                                  
 Conv/BatchNorm (BatchNormaliza  (None, 112, 112, 16  64         ['Conv[0][0]']                   
 tion)                          )                                                           

In [21]:
import numpy as np
import cv2
import time
import tensorflow as tf
import pandas as pd
from sklearn.preprocessing import StandardScaler

image_path = "dataset/ready_to_train/test/1/A000801-1-1.npy"
target_id = "A000801"
exclude_cols = ["to_patient_id", "is_icu"]
df = pd.read_csv("dataset/" + "imputed.csv")

numeric_cols = df.drop(columns=exclude_cols).select_dtypes(include=["number"]).columns
tabular_continuous_cols = [col for col in numeric_cols if df[col].nunique() > 5]
tabular_categorical_cols = [col for col in numeric_cols if col not in tabular_continuous_cols]
scaler = StandardScaler()
scaler.fit(df[tabular_continuous_cols])
feature_means = scaler.mean_
feature_stds = scaler.scale_
# 1. Clinic Feature Processing (Logic from your DataGenerator)
patient_data = df[df["to_patient_id"] == target_id]
# Ensure numeric_cols is defined based on your specific exclude list

ordered_processed = np.zeros((1, len(numeric_cols)))

# Normalize Continuous Data
cont_data = patient_data[tabular_continuous_cols].values[:1]
cont_data = (cont_data - feature_means) / feature_stds 

# Get Categorical Data
cat_data = patient_data[tabular_categorical_cols].values[:1]

# Map to ordered array (60 dimensions)
for i, col in enumerate(numeric_cols):
    if col in tabular_continuous_cols:
        idx = tabular_continuous_cols.index(col)
        ordered_processed[0, i] = cont_data[0, idx]
    else:
        idx = tabular_categorical_cols.index(col)
        ordered_processed[0, i] = cat_data[0, idx]

clinic_input = ordered_processed.astype('float32')

# 2. Image Processing (FIXED for CV_64F error)
img = np.load(image_path).astype('float32') # Cast here to avoid OpenCV depth error

if img.shape[:2] != (224, 224):
    img = cv2.resize(img, (224, 224))

# Convert grayscale to RGB if it only has 2 dimensions
if len(img.shape) == 2: 
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
elif img.shape[2] == 1: # Handle (224, 224, 1) cases
    img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)

img_input = np.expand_dims(img / 255.0, axis=0)

# 3. Create Intermediate Feature Extractor (576 dims)
# Stops at the layer before the final classification head
feature_extractor = tf.keras.Model(
    inputs=imaging_model.input, 
    outputs=imaging_model.layers[-2].output
)

In [23]:
import time
import numpy as np

def benchmark_all_components(img_in, clinic_in, iterations=50):
    """
    Benchmarks each model component and the end-to-end merged model.
    Returns mean and std dev in ms/sample.
    """
    # 1. Prepare Storage
    results = {
        "Imaging (CNN)": [],
        "Clinic (Tabular)": [],
        "Merged (End-to-End)": []
    }
    
    # 2. Warm-up (Important: Initial graph tracing is slow)
    for _ in range(5):
        _ = imaging_model.predict(img_in, verbose=0)
        _ = clinic_model.predict(clinic_in, verbose=0)
        _ = end_to_end_model.predict([img_in, clinic_in], verbose=0)

    # 3. Benchmark Loop
    for _ in range(iterations):
        # --- Individual Imaging ---
        start = time.perf_counter()
        _ = imaging_model.predict(img_in, verbose=0)
        results["Imaging (CNN)"].append((time.perf_counter() - start) * 1000)
        
        # --- Individual Clinic ---
        start = time.perf_counter()
        _ = clinic_model.predict(clinic_in, verbose=0)
        results["Clinic (Tabular)"].append((time.perf_counter() - start) * 1000)
        
        # --- Merged Pipeline ---
        start = time.perf_counter()
        _ = end_to_end_model.predict([img_in, clinic_in], verbose=0)
        results["Merged (End-to-End)"].append((time.perf_counter() - start) * 1000)

    # 4. Print Results with +/- (Standard Deviation)
    print(f"{'Component':<20} | {'Mean Latency':<15} | {'Std Dev':<10}")
    print("-" * 50)
    
    for name, latencies in results.items():
        mean_val = np.mean(latencies)
        std_val = np.std(latencies)
        print(f"{name:<20} | {mean_val:>8.2f} ms      | ± {std_val:>5.2f} ms")

# Run the benchmark
benchmark_all_components(img_input, clinic_input, iterations=50)

Component            | Mean Latency    | Std Dev   
--------------------------------------------------
Imaging (CNN)        |    23.83 ms      | ±  1.97 ms
Clinic (Tabular)     |    18.98 ms      | ±  1.91 ms
Merged (End-to-End)  |    26.03 ms      | ±  3.13 ms
